<a href="https://colab.research.google.com/github/akimotolab/CMAES_Tutorial/blob/main/6_advanced_adaptation_mechanisms.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 高级协方差矩阵自适应机制

为了便于理解，前面的章节主要使用最容易解释的 rank-$\mu$ update 来介绍 CMA-ES。现代 CMA-ES（例如 dd-CMA-ES [1]）会组合 rank-one update [2]、rank-$\mu$ update [3]、active update [4] 和 diagonal acceleration [1]，从而更高效地学习搜索分布。

相对于基础 rank-$\mu$ update，各组件的主要优势是：
* **rank-one update**：种群较小时尤其有效，并且擅长学习协方差矩阵中应当增大的特征值方向。
* **active update**：利用劣质候选解的信息主动压缩不希望搜索的方向，擅长更快学习较小的特征值。
* **diagonal acceleration**：专门加速每个坐标方向方差的学习。

本章逐一解释这些机制，并最终把它们组合成 dd-CMA-ES。

[1] Y. Akimoto, N. Hansen; Diagonal Acceleration for Covariance Matrix Adaptation Evolution Strategies. Evol Comput 2020; 28 (3): 405–435. https://doi.org/10.1162/evco_a_00260

[2] N. Hansen and A. Ostermeier, Completely Derandomized Self-Adaptation in Evolution Strategies, Evolutionary Computation 9(2), 2001.

[3] Hansen N, Müller SD, Koumoutsakos P. Reducing the time complexity of the derandomized evolution strategy with covariance matrix adaptation, 2003.

[4] G. A. Jastrebski and D. V. Arnold, Improving Evolution Strategies through Active Covariance Matrix Adaptation, CEC 2006.

## dd-CMA-ES 代码

下面给出包含本章各更新机制的 dd-CMA-ES 实现。更完整的实现（含重启、约束处理等）可参考：
* https://gist.github.com/youheiakimoto/1180b67b5a0b1265c204cba991fa8518
* https://github.com/akimotolab/multi-fidelity

为避免把算法说明淹没在重复的工程辅助代码中，本中文版本保留核心 dd-CMA 更新器，并在实验中使用轻量终止逻辑。算法更新公式与原教程一致。

In [ ]:
import warnings
import math
import numpy as np
import matplotlib.pyplot as plt

class DdCma:
    """dd-CMA：带 diagonal decoding 的 CMA-ES。"""
    def __init__(self, xmean0, sigma0, lam=None,
                 flg_covariance_update=True,
                 flg_variance_update=True,
                 flg_active_update=True,
                 flg_force_correlation=None,
                 beta_eig=None, beta_thresh=2.):
        self.N = len(xmean0)
        self.chiN = np.sqrt(self.N) * (1.0 - 1.0/(4.0*self.N) + 1.0/(21.0*self.N*self.N))
        self.flg_covariance_update = flg_covariance_update
        self.flg_variance_update = flg_variance_update
        self.flg_active_update = flg_active_update
        self.flg_force_correlation = flg_variance_update if flg_force_correlation is None else flg_force_correlation
        self.beta_eig = beta_eig if beta_eig else 10. * self.N
        self.beta_thresh = beta_thresh
        self.lam = lam if lam else 4 + int(3 * math.log(self.N))
        assert self.lam > 2
        w = math.log((self.lam + 1)/2.0) - np.log(np.arange(1,self.lam+1))
        w[w>0] /= np.sum(np.abs(w[w>0])); w[w<0] /= np.sum(np.abs(w[w<0]))
        self.mueff_positive = 1./np.sum(w[w>0]**2)
        self.mueff_negative = 1./np.sum(w[w<0]**2)
        self.cm = 1.
        self.cs = (self.mueff_positive + 2.)/(self.N + self.mueff_positive + 5.)
        self.ds = 1. + self.cs + 2.*max(0., math.sqrt((self.mueff_positive-1.)/(self.N+1.))-1.)
        expo = 0.75
        mu_prime = self.mueff_positive + 1./self.mueff_positive - 2. + self.lam/(2.*self.lam+10.)
        m = self.N*(self.N+1)/2
        self.cone = 1./(2*(m/self.N+1.)*(self.N+1.)**expo + self.mueff_positive/2.)
        self.cmu = min(1.-self.cone, mu_prime*self.cone)
        self.cc = math.sqrt(self.mueff_positive*self.cone)/2.
        self.w = np.array(w)
        self.w[w<0] *= min(1.+self.cone/self.cmu, 1.+2.*self.mueff_negative/(self.mueff_positive+2.))
        m = self.N
        self.cdone = 1./(2*(m/self.N+1.)*(self.N+1.)**expo + self.mueff_positive/2.)
        self.cdmu = min(1.-self.cdone, mu_prime*self.cdone)
        self.cdc = math.sqrt(self.mueff_positive*self.cdone)/2.
        self.wd = np.array(w)
        self.wd[w<0] *= min(1.+self.cdone/self.cdmu, 1.+2.*self.mueff_negative/(self.mueff_positive+2.))
        self.xmean = np.array(xmean0); self.D = np.array(sigma0); self.sigma = 1.
        self.C = np.eye(self.N); self.S = np.ones(self.N); self.B = np.eye(self.N)
        self.sqrtC = np.eye(self.N); self.invsqrtC = np.eye(self.N); self.Z = np.zeros((self.N,self.N))
        self.pc = np.zeros(self.N); self.pdc = np.zeros(self.N); self.ps = np.zeros(self.N)
        self.pc_factor = self.pdc_factor = self.ps_factor = 0.
        self.teig = max(1,int(1./(self.beta_eig*(self.cone+self.cmu))))
        self.neval = 0; self.t = 0; self.beta = 1.
        self.arf = np.zeros(self.lam); self.arx = np.zeros((self.lam,self.N))

    def sample(self):
        arz = np.random.randn(self.lam,self.N)
        ary = np.dot(arz,self.sqrtC) if self.flg_covariance_update else arz
        arx = ary*(self.D*self.sigma)+self.xmean
        return arx,ary,arz

    def update(self, idx, arx, ary, arz):
        w=self.w; wc=self.w; wd=self.wd; sarz=arz[idx]; sary=ary[idx]
        dz=np.dot(w[w>0],sarz[w>0]); dy=np.dot(w[w>0],sary[w>0])
        self.xmean += self.cm*self.sigma*self.D*dy
        # CSA 步长更新
        self.ps_factor=(1-self.cs)**2*self.ps_factor+self.cs*(2-self.cs)
        self.ps=(1-self.cs)*self.ps+math.sqrt(self.cs*(2-self.cs)*self.mueff_positive)*dz
        normsquared=np.sum(self.ps*self.ps)
        hsig=normsquared/self.ps_factor/self.N < 2.0+4.0/(self.N+1)
        self.sigma *= math.exp((math.sqrt(normsquared)/self.chiN-math.sqrt(self.ps_factor))*self.cs/self.ds)
        # 完整协方差 C：rank-mu + rank-one
        if self.flg_covariance_update:
            if self.cmu==0: rank_mu=0.
            elif self.flg_active_update:
                rank_mu=np.dot(sarz[wc>0].T*wc[wc>0],sarz[wc>0])-np.sum(wc[wc>0])*np.eye(self.N)
                rank_mu+=np.dot(sarz[wc<0].T*(wc[wc<0]*self.N/np.linalg.norm(sarz[wc<0],axis=1)**2),sarz[wc<0])-np.sum(wc[wc<0])*np.eye(self.N)
            else:
                rank_mu=np.dot(sarz[wc>0].T*wc[wc>0],sarz[wc>0])-np.sum(wc[wc>0])*np.eye(self.N)
            if self.cone==0: rank_one=0.
            else:
                self.pc=(1-self.cc)*self.pc+hsig*math.sqrt(self.cc*(2-self.cc)*self.mueff_positive)*self.D*dy
                self.pc_factor=(1-self.cc)**2*self.pc_factor+hsig*self.cc*(2-self.cc)
                zpc=np.dot(self.pc/self.D,self.invsqrtC)
                rank_one=np.outer(zpc,zpc)-self.pc_factor*np.eye(self.N)
            self.Z += self.cmu*rank_mu+self.cone*rank_one
        # diagonal decoding D
        if self.flg_variance_update:
            self.pdc=(1-self.cdc)*self.pdc+hsig*math.sqrt(self.cdc*(2-self.cdc)*self.mueff_positive)*self.D*dy
            self.pdc_factor=(1-self.cdc)**2*self.pdc_factor+hsig*self.cdc*(2-self.cdc)
            DD=self.cdone*(np.dot(self.pdc/self.D,self.invsqrtC)**2-self.pdc_factor)
            if self.flg_active_update:
                DD += self.cdmu*np.dot(wd[wd>0],sarz[wd>0]**2)
                DD += self.cdmu*np.dot(wd[wd<0]*self.N/np.linalg.norm(sarz[wd<0],axis=1)**2,sarz[wd<0]**2)
                DD -= self.cdmu*np.sum(wd)
            else:
                DD += self.cdmu*np.dot(wd[wd>0],sarz[wd>0]**2)-self.cdmu*np.sum(wd[wd>0])
            self.beta = 1/max(1,np.max(self.S)/np.min(self.S)-self.beta_thresh+1.) if self.flg_covariance_update else 1.
            self.D *= np.exp((self.beta/2)*DD)
        # 定期将累积更新写回 C 并重新分解
        if self.flg_covariance_update and (self.t+1)%self.teig==0:
            eig=np.linalg.eigvalsh(self.Z); fac=min(0.75/abs(eig.min()),1.)
            self.C=np.dot(np.dot(self.sqrtC,np.eye(self.N)+fac*self.Z),self.sqrtC)
            if self.flg_force_correlation:
                cd=np.sqrt(np.diag(self.C)); self.D*=cd; self.C=(self.C/cd).T/cd
            DD,self.B=np.linalg.eigh(self.C); self.S=np.sqrt(DD)
            self.sqrtC=np.dot(self.B*self.S,self.B.T); self.invsqrtC=np.dot(self.B/self.S,self.B.T); self.Z[:,:]=0.

    def onestep(self, func):
        arx,ary,arz=self.sample(); arf=func(arx); self.neval+=len(arf); idx=np.argsort(arf)
        if not np.all(arf[idx[1:]]-arf[idx[:-1]]>0.): warnings.warn('assumed no tie, but there exists',RuntimeWarning)
        self.update(idx,arx,ary,arz); self.t+=1; self.arf=arf; self.arx=arx

    @property
    def coordinate_std(self):
        return self.sigma*self.D*np.sqrt(np.diag(self.C)) if self.flg_covariance_update else self.sigma*self.D


#### 协方差矩阵的分解表示

dd-CMA 将搜索分布的协方差写成
$$\Sigma=\sigma^2DCD.$$
其中 $\sigma^2$ 是全局步长，由 CSA 更新；$D$ 是对角矩阵，$\sigma[D]_{i,i}$ 对应第 $i$ 个坐标的标准差尺度；$C$ 的对角元素保持为 1，主要表达变量之间的相关结构。

可以把 Separable-CMA 理解为固定 $C=I$、只学习 $D$；传统完整 CMA 则相当于把 $DCD$ 作为整体学习；diagonal acceleration 的关键是将 $D$ 与 $C$ 分开，用不同学习速度自适应。

采样步骤为：
1. $z_i\sim\mathcal{N}(0,I)$；
2. $y_i=\sqrt C z_i$；
3. $x_i=m+\sigma D y_i$。

#### 只使用 rank-$\mu$ update 时

若固定 $D$、只更新 $C$，基础 rank-$\mu$ 更新为
$$C\leftarrow C+c_\mu\sum_{i=1}^{\lambda}w_i(y_{i:\lambda}y_{i:\lambda}^\mathrm{T}-C),$$
其中基础版本假设 $w_i\ge0$ 且权重和为 1。

若固定 $C=I$、只学习 $D$，可以在对数尺度更新
$$\log D^2\leftarrow\log D^2+c_{\mu,D}\sum_{i=1}^{\lambda}w_i\operatorname{diag}(z_{i:\lambda}z_{i:\lambda}^\mathrm{T}-I).$$
对指数函数做一阶近似后，这与前面 Separable-CMA 的方差更新对应。

## Rank-one update

rank-$\mu$ update 每代只利用当前优秀的 $\mu$ 个候选解，而 rank-one update 的目标是通过进化路径累积跨多代的方向信息，因此在 $\mu$ 较小、种群较小时尤其有效。它还能够利用方向的**符号一致性**，这是外积型 rank-$\mu$ 更新无法直接利用的信息。

用于 $C$ 的进化路径可写为
$$p_C\leftarrow(1-c_C)p_C+\sqrt{\frac{c_C(2-c_C)}{\sum_jw_j^2}}\sum_iw_i y_{i:\lambda},$$
用于对角尺度 $D$ 的路径类似：
$$p_D\leftarrow(1-c_D)p_D+\sqrt{\frac{c_D(2-c_D)}{\sum_jw_j^2}}\sum_iw_i z_{i:\lambda}.$$
然后分别向 $C$、$D$ 更新中加入
$$c_1(p_Cp_C^\mathrm{T}-C),\qquad c_{1,D}\operatorname{diag}(p_Dp_D^\mathrm{T}-I).$$
在随机目标函数下，路径仍满足与 CSA 类似的无偏性，因此新增项期望为 0；如果跨多代持续选择同一方向，路径就会沿该方向积累，使对应协方差特征值增大。

符号信息的意义可以这样看：rank-$\mu$ 对 $y$ 与 $-y$ 的外积完全相同，所以无法区分“连续同向”与“正负交替”；rank-one 先累积向量本身，因此 $y,-y$ 会互相抵消，而 $y,y$ 会叠加。这使算法能够识别真正持续的搜索方向。

#### 验证实验

使用 $d/2$-tablet 类函数：一半变量的敏感度与另一半相差很大。比较仅 rank-$\mu$ 与 rank-$\mu$+rank-one。默认较小种群下两者学习协方差的过程会明显不同；当 `lam` 很大时，rank-$\mu$ 本身拥有足够多的每代样本，rank-one 的额外收益会减弱。

In [ ]:
N=20
def half_tablet(x):
    k=x.shape[1]//2
    return np.sum(x[:,:k]**2,axis=1)+1e6*np.sum(x[:,k:]**2,axis=1)

def run_rank_one(enable=True, lam=None):
    cma=DdCma(np.random.randn(N),np.ones(N)*2.,lam=lam,flg_covariance_update=True,flg_variance_update=False,flg_active_update=False)
    if not enable: cma.cone=0
    hist=[]
    while cma.neval<1e6:
        cma.onestep(half_tablet); hist.append(np.min(cma.arf))
        if hist[-1]<=1e-8: break
    return cma,np.asarray(hist)

cma_mu,h_mu=run_rank_one(False)
cma_one,h_one=run_rank_one(True)
plt.semilogy(h_mu,label='rank-mu'); plt.semilogy(h_one,label='rank-mu + rank-one'); plt.grid(); plt.legend()

#### 行为差异的解释

这个测试函数的 Hessian 有 $d/2$ 个 $10^6$ 特征值和 $d/2$ 个 1，因此理想协方差也需要形成相差约 $10^6$ 的两组尺度。仅 rank-$\mu$ 时，较小的一组协方差特征值会一起下降；加入 rank-one 后，除了这些小特征值下降，还会看到较大的特征值沿着进化路径方向一个接一个更快增大。

这是 rank-one 项 $p_Cp_C^\mathrm{T}-C$ 的直接结果：$-C$ 主要做整体缩放，而 $p_Cp_C^\mathrm{T}$ 是 rank-1 矩阵，只沿当前进化路径增强一个方向。随着不同低敏感方向先后成为主要搜索方向，这些方向对应的特征值会逐个被扩张。

当协方差矩阵还没有很好逼近 Hessian 的逆时，CSA 往往会把 $\sigma$ 调小，因为变换后搜索空间仍然病态，连续均值移动无法保持足够长的同向路径。事实上，此时理论最优步长本身也会更小，因此这种现象并不是 CSA 独有。关于一般凸二次函数上的步长与收敛率可参考 Akimoto et al. (TCS 2020) 与 Morinaga et al. (IEEE TEC / arXiv:2103.01578)。

## Active update

基础 rank-$\mu$ 更新使用非负权重：
$$C\leftarrow(1-c_\mu)C+c_\mu\sum_iw_i y_{i:\lambda}y_{i:\lambda}^\mathrm{T}.$$
优秀解只能主动增大相应方向的协方差；想缩小某一个方向，通常只能依靠整体 $(1-c_\mu)$ 缩放，再同时补偿其余方向，因此学习小特征值相对缓慢。rank-one 也有类似特点。

Active update 的核心非常直接：给排名较差的候选解赋予**负权重**。这些解集中在哪些方向，说明哪些方向更容易产生较差目标值，因此应主动减小这些方向的搜索方差。若优秀解权重和为 1、劣质解负权重和为 -1，可写成
$$C\leftarrow C+c_\mu\sum_{i=1}^{\mu}w_i y_i y_i^\mathrm{T}-c_\mu\sum_{i=\mu+1}^{\lambda}|w_i|y_i y_i^\mathrm{T}.$$
这样协方差不再只能依靠整体衰减来变小，而是可以沿劣质解方向主动收缩。实际实现还需要限制负更新强度，保证协方差保持正定。

#### 验证实验
用两个条件数很高的函数比较 active update：
* Cigar：$f(x)=x_1^2+10^6\sum_{i=2}^{d}x_i^2$。主要需要把一个低敏感方向的协方差变大，因此 active update 收益有限。
* Tablet：$f(x)=10^6x_1^2+\sum_{i=2}^{d}x_i^2$。主要需要快速压小一个高敏感方向的协方差，因此 active update 通常明显加速。

In [ ]:
def cigar(x): return x[:,0]**2+1e6*np.sum(x[:,1:]**2,axis=1)
def tablet(x): return 1e6*x[:,0]**2+np.sum(x[:,1:]**2,axis=1)
def compare_active(fobj,N=20):
    curves={}
    for flag in [True,False]:
        cma=DdCma(np.random.randn(N),np.ones(N)*2.,flg_covariance_update=True,flg_variance_update=False,flg_active_update=flag)
        h=[]
        while cma.neval<1e6:
            cma.onestep(fobj); h.append(np.min(cma.arf))
            if h[-1]<=1e-8: break
        curves[flag]=h
    return curves
curves=compare_active(tablet)
plt.semilogy(curves[True],label='active'); plt.semilogy(curves[False],label='without active'); plt.grid(); plt.legend()

## Diagonal Acceleration

比较 Separable-CMA（固定 $C=I$、只更新 $D$）与标准 CMA（固定 $D$、只更新完整 $C$），前者有两个优势：
1. 每代内部计算和内存关于维数 $d$ 近似线性，而完整 CMA 通常为二次量级，因此高维时更便宜。
2. 对角尺度 $D$ 可以使用远高于完整相关矩阵 $C$ 的学习率。典型推荐量级中，$D$ 的学习率约为 $1/d$，而 $C$ 的学习率约为 $1/d^2$。所以如果问题主要只是不同变量尺度不同，Separable-CMA 会更快。

这里主要关注第二个优势，以及如何在不牺牲旋转问题能力的前提下保留这种快速尺度学习。

#### 对比实验

Separable-Ellipsoid 定义为
$$f(x)=\sum_{i=1}^{d}10^{6\frac{i-1}{d-1}}x_i^2,$$
其 Hessian 为对角矩阵。Rotated-Ellipsoid 则通过随机正交矩阵 $R$ 做坐标旋转，写成 $f(Rx)$，对应 Hessian 为 $R^\mathrm{T}D_{ell}R$，不再对角。

在前者上，Separable-CMA 因为可以用较大学习率快速学到每个坐标尺度，通常比完整 CMA 更快；维数越高，这一差异越明显。
在旋转后的问题上，Separable-CMA 无法表示相关方向，因此会退化得非常严重，而完整 CMA 基本保持与未旋转问题相似的行为。

In [ ]:
def random_axes(dim,n_axes):
    R=np.random.normal(0,1,(n_axes,dim))
    for i in range(n_axes):
        for j in range(i): R[i]-=np.dot(R[i],R[j])*R[j]
        R[i]/=np.linalg.norm(R[i])
    return R
def separable_ellipsoid(x):
    a=1e3; dim=x.shape[1]; d=a**(2.0*np.arange(dim)/float(dim-1)); return np.dot(x**2,d)
N=20; R=random_axes(N,N)
def rotated_ellipsoid(x): return separable_ellipsoid(np.dot(x,R))

#### 为什么 Separable-CMA 在旋转问题上很慢

根本原因是：旋转 Ellipsoid 的理想预条件矩阵接近 Hessian 的逆 $R^\mathrm{T}D_{ell}^{-1}R$，它无法被对角矩阵充分近似。不论怎样调整对角 $D$，变换后的二次型仍然保持很高条件数；而当搜索协方差近似单位阵时，收敛速度会强烈受条件数限制。因此只学习坐标尺度无法解决变量耦合造成的病态性。

#### Diagonal Decoding 的动机与思路

Separable-CMA 擅长快速学习各变量尺度，但无法预先知道真实问题是否接近可分；标准 CMA 对旋转问题稳健，却需要更慢地学习完整矩阵。dd-CMA 同时让 $D$ 学习坐标尺度、让 $C$ 学习变量相关性：如果问题近似可分，快速的 $D$ 更新起主要作用；如果问题强耦合，$C$ 最终吸收旋转结构。

不过不能简单地把两套更新机械叠加。当 $C$ 已经高度病态时，$D$ 的微小变化也可能让新旧分布之间的 KL 距离过大。dd-CMA 因此通过系数 $\beta$ 动态限制 $D$ 的有效学习率，使每次 diagonal decoding 更新造成的分布变化保持受控。

#### dd-CMA 的预期行为
* 在 Separable-Ellipsoid 上，效率接近 Separable-CMA。
* 在 Rotated-Ellipsoid 上，效率接近标准完整 CMA。

这正是 diagonal acceleration 的目标：在不知道问题是否可分的前提下，同时获得快速尺度学习和对旋转/相关结构的鲁棒性。

In [ ]:
def run_ddcma(fobj,N=20,max_neval=50000):
    cma=DdCma(np.random.randn(N),np.ones(N)*2.,flg_covariance_update=True,flg_variance_update=True,flg_active_update=True)
    hist=[]
    while cma.neval<max_neval:
        cma.onestep(fobj); hist.append(np.min(cma.arf))
        if hist[-1]<=1e-8: break
    return cma,np.asarray(hist)
_,h_sep=run_ddcma(separable_ellipsoid)
_,h_rot=run_ddcma(rotated_ellipsoid)
plt.semilogy(h_sep,label='separable'); plt.semilogy(h_rot,label='rotated'); plt.grid(); plt.legend()